# Classifiers, pipelines and the confusion matrix

This notebook covers the first block of Session 8. In Session 6 you fitted a logistic regression for churn; in Session 7 you learned to validate models honestly. Now we treat classification systematically: what a classification task is and which baselines a model must beat, how to turn a real table with text categories and numbers into model input, how `Pipeline` and `ColumnTransformer` keep that preparation leak-free, and a second classifier, k-nearest neighbours, whose decision boundary makes underfitting and overfitting visible. The last three sections introduce the confusion matrix and the metrics derived from it (precision, recall, F1 and macro-F1), which show *which* errors a classifier makes.

The code cells build on each other; run them in order. The first cell moves to the repository root, where the case-study paths start. They use the IBM Telco churn data (7,043 customers, 26.5 % churners), downloaded from IBM's GitHub repository.

```mermaid
flowchart LR
    R["Raw table:<br/>numbers + categories"] --> CT["ColumnTransformer"]
    CT --> N["numeric columns:<br/>impute, scale"]
    CT --> C["categorical columns:<br/>one-hot encode"]
    N --> M["model:<br/>k-NN or logistic regression"]
    C --> M
    M --> P["predicted class<br/>and probability"]
    P --> E["confusion matrix:<br/>precision, recall, F1"]
```

In [1]:
# Run the code from the repository root, where the case-study paths start.
import os
from pathlib import Path

os.chdir(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").is_dir()))

## Classification tasks and baselines

**Concept.** A **classification** task predicts a category, the **class**, from features. With two classes (churn yes/no, spam/not spam) it is **binary**; with more (the room type of an Airbnb listing, or the product heading of a customs decision in Session 13) it is **multiclass**. By convention the class of interest, usually the rarer one, is called the **positive** class and coded 1; here, "churns".

Before any model, define **baselines**: simple predictions that a model must beat to be worth its cost.

- The **majority-class** baseline predicts the most frequent class for everyone. With 26.5 % churners, "nobody churns" is right for 73.5 % of customers. Accuracy below 0.735 is worse than doing nothing.
- A **hand-made rule** encodes domain knowledge, for example "customers on a month-to-month contract in their first year will churn". It is cheap, explainable and often surprisingly strong.

Worked example: of 10 customers, 3 churn. "Nobody churns" is correct for 7 of 10: accuracy 0.7, yet it finds none of the churners. A rule that flags 4 customers, 2 of whom churn, is correct for 2 churners + 5 of the 7 loyal customers = 7 customers as well: accuracy 0.7 again, but it finds 2 of 3 churners. Accuracy alone does not separate the two; the last three sections of this notebook introduce metrics that do.

**Why it matters.** Without a baseline a number such as "79 % accuracy" means nothing. In imbalanced problems a model can look good while only predicting the majority class. Baselines also set expectations for the business: how much better than the current rule of thumb is the model?

**How it works in Python.** `DummyClassifier` implements the majority-class baseline with the usual `fit`/`predict`/`score` interface:

In [2]:
import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, recall_score
from sklearn.model_selection import train_test_split

URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
telco = pd.read_csv(URL)
telco["TotalCharges"] = pd.to_numeric(telco["TotalCharges"], errors="coerce")   # 11 blanks -> NaN
y = (telco["Churn"] == "Yes").astype(int)
X = telco.drop(columns=["customerID", "Churn"])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)

dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
print(round(dummy.score(X_test, y_test), 3))                       # 0.735: "nobody churns"

rule = ((X_test["Contract"] == "Month-to-month") & (X_test["tenure"] < 12)).astype(int)
print(round(accuracy_score(y_test, rule), 3), round(recall_score(y_test, rule), 3))   # 0.756 0.555

0.735
0.756 0.555


The rule beats the majority baseline slightly in accuracy and, unlike it, finds 55.5 % of the churners.

**In practice.**
- Telecom and subscription businesses have long used rule-based churn flags (contract type, tenure, complaints) in customer relationship management systems; a churn model is judged against these rules.
- E-mail spam filtering started with hand-written rules (SpamAssassin's scored rule sets) before learned classifiers were added on top; the rules remain a baseline.
- Benchmarks and competitions publish a trivial baseline (the most frequent class, or a simple rule) next to the leaderboard, so that every submission can be judged against it.

> **Warning.** On imbalanced data, a high accuracy can mean nothing more than "predicts the majority class". Always report the majority-class baseline next to your model.

## Preparing inputs: one-hot and ordinal encoding, scaling

**Concept.** Most models need numbers. The Telco table has 4 numeric columns and 15 text columns such as `Contract` ("Month-to-month", "One year", "Two year") or `PaymentMethod`.

- **One-hot encoding** turns a column with *m* categories into *m* columns of 0/1, one per category. "One year" becomes (0, 1, 0). It imposes no order and suits **nominal** categories (payment method, internet service).
- **Ordinal encoding** maps the categories to integers 0, 1, 2, … in a given order. It suits **ordinal** categories with a meaningful order (contract length, education level, clothing sizes) and keeps one column. Used on nominal categories, it invents an order and distances that do not exist.
- **Scaling.** Distance-based models (k-NN) and penalised models (ridge, lasso, logistic regression) are sensitive to units: tenure ranges 0–72, total charges 0–8,700. **Standardisation** (`StandardScaler`) subtracts the training mean and divides by the training standard deviation, so every numeric feature has mean 0 and SD 1. Tree-based models (Session 10) do not need scaling.

Worked example of standardisation: tenure values 2, 10, 30 have mean 14 and SD (population) ≈ 11.8; the standardised values are (2 − 14)/11.8 ≈ −1.02, −0.34 and 1.36.

**Why it matters.** Encoding choices change what the model can learn. Unscaled features let the one with the largest numbers dominate a distance. A category that appears only in the test data (a new payment method) crashes a naive encoder; `handle_unknown="ignore"` encodes it as all zeros instead.

**How it works in Python.**

In [3]:
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

oh = OneHotEncoder(sparse_output=False).fit(X_train[["Contract"]])
print(oh.get_feature_names_out())          # ['Contract_Month-to-month' 'Contract_One year' 'Contract_Two year']
print(oh.transform(X_test[["Contract"]].head(3)))
# [[1. 0. 0.]   month-to-month
#  [1. 0. 0.]   month-to-month
#  [0. 0. 1.]]  two year

order = [["Month-to-month", "One year", "Two year"]]
ordinal = OrdinalEncoder(categories=order).fit(X_train[["Contract"]])
print(ordinal.transform(pd.DataFrame({"Contract": ["Two year", "Month-to-month"]})).ravel())   # [2. 0.]

scaler = StandardScaler().fit(X_train[["tenure"]])
print(scaler.mean_.round(1), scaler.scale_.round(1))          # [32.1] [24.5]: training mean and SD of tenure
print(scaler.transform(pd.DataFrame({"tenure": [0, 72]})).ravel().round(2))   # [-1.31  1.63] in SD units

['Contract_Month-to-month' 'Contract_One year' 'Contract_Two year']
[[1. 0. 0.]
 [1. 0. 0.]
 [0. 0. 1.]]
[2. 0.]
[32.1] [24.5]
[-1.31  1.63]


**In practice.**
- Credit scoring: categorical attributes such as employment type or housing status are one-hot encoded (or grouped into bins) before a logistic regression scorecard is fitted.
- Survey and HR analytics: ordinal answers (Likert scales, education levels) are often ordinal-encoded to keep their order.

> **Caution.** Fit encoders and scalers on the **training data only**. The scaler's mean and SD are learned parameters; fitting them on all rows leaks information from the test set (Session 7). The next section makes this automatic.

> **Tip.** A column with thousands of categories (product id, user id) explodes into thousands of one-hot columns. Session 9 shows alternatives such as target encoding, which must be fitted inside the cross-validation.

> 📖 **Read more:** [extras/01-categorical-encoding.ipynb](extras/01-categorical-encoding.ipynb) (INRIA scikit-learn MOOC): ordinal and one-hot encoding of the adult census data, and what each does to a linear model.
>
> 📖 **Read more:** [extras/02-scaling-in-pipelines.ipynb](extras/02-scaling-in-pipelines.ipynb) (INRIA scikit-learn MOOC): `StandardScaler` with `fit` and `transform`, a first pipeline and `cross_validate`, with diagrams of what each call learns.

## Pipeline and ColumnTransformer

**Concept.** A **`ColumnTransformer`** applies different preparation to different columns and concatenates the results: for example, impute and scale the numeric columns, one-hot encode the categorical ones. A **`Pipeline`** puts the model after it. Fitting the pipeline fits every step on the training data only; predicting applies the stored steps to new data. Because the pipeline is a single estimator, `cross_val_score` and `GridSearchCV` refit the whole preparation inside every fold (Session 7, lecture notebook 01).

```mermaid
sequenceDiagram
    participant U as You
    participant P as Pipeline
    participant CT as ColumnTransformer
    participant M as Model
    U->>P: fit(X_train, y_train)
    P->>CT: fit_transform(X_train)
    CT-->>P: medians, means, SDs, categories learned
    P->>M: fit(transformed X_train, y_train)
    U->>P: predict(X_test)
    P->>CT: transform(X_test) with stored values
    P->>M: predict(transformed X_test)
    M-->>U: classes
```

**Why it matters.** Real tables mix numbers and categories and contain missing values. A pipeline keeps preparation and model together, so that exactly the same steps are applied to training data, test data and, later, new customers in production (Session 16). It prevents leakage by construction and makes the whole workflow tunable: `columntransformer__num__simpleimputer__strategy` is a hyperparameter like any other.

**How it works in Python.**

In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

numeric = ["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen"]
categorical = [c for c in X.columns if c not in numeric]        # 15 text columns
pre = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), numeric),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical),
])
logreg = make_pipeline(pre, LogisticRegression(max_iter=1000))
logreg.fit(X_train, y_train)                                    # every step learns from X_train only

print(round(logreg.score(X_test, y_test), 3))                   # 0.792 accuracy
print(logreg[:-1].transform(X_test).shape)                      # (1761, 45): 4 numeric + 41 one-hot columns
print(logreg.predict_proba(X_test)[:3, 1].round(2))             # [0.48 0.12 0.  ]: churn probabilities

0.792
(1761, 45)
[0.48 0.12 0.  ]


The INRIA worked example [02-column-transformer.ipynb](02-column-transformer.ipynb) builds the same structure step by step.

**In practice.**
- Customer analytics teams deploy fitted pipelines that score the whole customer base every week; the pipeline guarantees that the scoring data are prepared like the training data.
- Hospitals and research groups share fitted prediction pipelines so that other sites can validate them on their own patients without re-implementing the preparation.

> **Warning.** `TotalCharges` contains 11 blank strings for new customers. `pd.to_numeric(..., errors="coerce")` turns them into `NaN`; without the `SimpleImputer` in the pipeline, logistic regression and k-NN raise an error.

## k-nearest neighbours and the decision boundary

**Concept.** **k-nearest neighbours** (k-NN) classifies a new case by looking at the *k* training cases closest to it (by Euclidean distance on the prepared features) and predicting their majority class. The share of churners among the *k* neighbours is its predicted probability. k-NN learns no coefficients; "fitting" just stores the training data.

The **decision boundary** is the line (or surface) in feature space where the prediction switches from one class to the other. Its shape depends on *k*:

- *k* = 1: every training point is its own neighbour. Training accuracy is close to 100 %, the boundary is jagged and follows every noisy point: **overfitting**.
- very large *k*: the prediction becomes the majority class almost everywhere; the boundary is smooth and misses real structure: **underfitting**.
- in between, the boundary is smooth but still follows the pattern; the validation score is highest.

*k* is a hyperparameter (Session 7): choose it by cross-validation.

Worked example: a new customer with standardised (tenure, monthly charges) = (−1.0, 0.8). Her five nearest training customers churned, stayed, churned, churned, stayed: 3 of 5 churned, so 5-NN predicts "churns" with probability 0.6.

![k-NN decision boundaries on two Telco features (tenure, monthly charges) for k = 1, 15 and 150. With k = 1 the churn region (orange) consists of many small islands; with k = 150 it is one smooth region of short-tenure, high-charge customers.](../figures/knn_boundaries.png)

**Why it matters.** k-NN is the simplest model that can draw any boundary, which makes it the clearest illustration of the trade-off between flexibility and stability. It also shows why scaling matters: without it, distances are dominated by `TotalCharges`. Its weaknesses (slow prediction on large data, poor performance with many irrelevant features) explain why other models are usually preferred in production.

**How it works in Python.** The same preprocessing, a different final step:

In [5]:
from sklearn.neighbors import KNeighborsClassifier

for k in [1, 5, 15, 50, 150, 500]:
    knn = make_pipeline(pre, KNeighborsClassifier(n_neighbors=k)).fit(X_train, y_train)
    print(k, round(knn.score(X_train, y_train), 3), round(knn.score(X_test, y_test), 3))
# k    train  test
# 1    0.997  0.726   <- memorises the training data: overfitting
# 5    0.84   0.756
# 15   0.815  0.775
# 50   0.804  0.784   <- best test accuracy, close to logistic regression (0.792)
# 150  0.797  0.778
# 500  0.795  0.776   <- smoother, slowly underfitting

1 0.997 0.726


5 0.84 0.756


15 0.815 0.775


50 0.804 0.784


150 0.797 0.778


500 0.795 0.776


In a real project, *k* would be chosen by cross-validation on the training data (`GridSearchCV` with `{"kneighborsclassifier__n_neighbors": [...]}`), not by the test score shown here for illustration. The scikit-learn gallery notebook [extras/03-knn-decision-boundary.ipynb](extras/03-knn-decision-boundary.ipynb) draws boundaries on the iris data with uniform and distance weights; the figure above is made by [figures/make_figures.py](../figures/make_figures.py).

**In practice.**
- Recommender systems use neighbour methods: Amazon's item-to-item collaborative filtering (Linden, Smith & York, 2003) recommends products similar to those a customer bought.
- Similarity search over embeddings (Sessions 14 and 15) is a nearest-neighbour search: retrieve the documents closest to a query.
- Anomaly detection (Session 11) often uses the distance to the *k*-th nearest neighbour as an outlier score.

> **Warning.** k-NN has to compare each new case with all stored training cases. With hundreds of thousands of rows and tens of thousands of features (for example word counts of texts), prediction becomes slow; use it on small tables or with approximate nearest-neighbour indexes.

> **Caution.** With many one-hot or irrelevant features, all points become roughly equally far apart (the "curse of dimensionality") and k-NN degrades. Select or reduce features first, or prefer logistic regression.

## Confusion matrix and accuracy

**Concept.** A **confusion matrix** counts the four outcomes of a binary prediction. With churn as the positive class:

| | predicted: stays (0) | predicted: churns (1) |
|---|---|---|
| **truly stays (0)** | true negative (TN) | false positive (FP) |
| **truly churns (1)** | false negative (FN) | true positive (TP) |

**Accuracy** = (TP + TN) / (TP + TN + FP + FN) is the share of correct predictions. A **false positive** (type I error) is a loyal customer wrongly flagged; a **false negative** (type II error) is a churner missed.

![Confusion matrix of the logistic regression on 1,761 Telco test customers at threshold 0.5: 1,146 true negatives, 148 false positives, 219 false negatives, 248 true positives.](../figures/confusion_matrix.png)

Worked example from the figure: accuracy = (1,146 + 248) / 1,761 = 0.792. But of the 467 churners, 219 (47 %) are missed.

**Why it matters.** Different errors have different costs. A retention campaign wastes money on false positives and loses customers through false negatives; a cancer screening test must above all avoid false negatives. The confusion matrix shows both errors separately and is the starting point of every other metric in this notebook.

**How it works in Python.**

In [6]:
from sklearn.metrics import confusion_matrix

knn = make_pipeline(pre, KNeighborsClassifier(n_neighbors=50)).fit(X_train, y_train)   # k = 50 from the loop above
pred = logreg.predict(X_test)                         # logistic regression, threshold 0.5
print(confusion_matrix(y_test, pred))
# [[1146  148]      rows = truth (stays, churns), columns = prediction
#  [ 219  248]]
tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
print(round((tp + tn) / (tp + tn + fp + fn), 3), round(accuracy_score(y_test, pred), 3))   # 0.792 0.792

[[1146  148]
 [ 219  248]]
0.792 0.792


`ConfusionMatrixDisplay.from_estimator(model, X_test, y_test)` draws the matrix; the scikit-learn gallery notebook [extras/04-confusion-matrix.ipynb](extras/04-confusion-matrix.ipynb) shows it with and without normalisation.

**In practice.**
- Medical screening reports the four cells as sensitivity (TP rate) and specificity (TN rate) of a test, for example for mammography or rapid antigen tests.
- Fraud teams at card issuers monitor the false-positive count closely, because every flagged transaction may block a legitimate customer or require an analyst's review.

> **Warning.** scikit-learn puts the **truth in rows** and the **prediction in columns**, with classes in sorted order (0 before 1). Some textbooks and tools use the transposed layout. Always label the axes.

## Precision, recall and F1

**Concept.**

- **Precision** = TP / (TP + FP): of the customers we flag, how many really churn? It measures how much of our effort is well spent.
- **Recall** (sensitivity, true positive rate) = TP / (TP + FN): of the customers who churn, how many do we flag? It measures how many cases we find.
- **F1** = 2 · precision · recall / (precision + recall), the harmonic mean. It is high only if both are high; a model with precision 1.0 and recall 0.01 has F1 ≈ 0.02.

Worked example from the confusion matrix: precision = 248 / (248 + 148) = 0.626; recall = 248 / (248 + 219) = 0.531; F1 = 2 · 0.626 · 0.531 / (0.626 + 0.531) = 0.575.

Precision and recall pull in opposite directions. Flagging more customers (a lower threshold) finds more churners (higher recall) but also flags more loyal ones (lower precision).

**Why it matters.** The two numbers answer two different business questions. A retention team with a limited budget cares about precision (do not waste offers); a regulator or a hospital may care first about recall (do not miss cases). F1 is a compromise when both matter and no costs are known; lecture notebook 04 replaces it with actual costs when they are known.

**How it works in Python.**

In [7]:
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score

for name, m in [("logistic regression", logreg), ("50-NN", knn)]:
    p = m.predict(X_test)
    print(name, [round(f(y_test, p), 3) for f in (precision_score, recall_score, f1_score)])
# logistic regression [0.626, 0.531, 0.575]
# 50-NN               [0.6, 0.552, 0.575]

print(classification_report(y_test, pred, target_names=["stays", "churns"], digits=3))
# per-class precision, recall, F1, support, plus macro and weighted averages

logistic regression [0.626, 0.531, 0.575]


50-NN [0.6, 0.552, 0.575]
              precision    recall  f1-score   support

       stays      0.840     0.886     0.862      1294
      churns      0.626     0.531     0.575       467

    accuracy                          0.792      1761
   macro avg      0.733     0.708     0.718      1761
weighted avg      0.783     0.792     0.786      1761



The two models have the same F1 but a different balance: k-NN finds slightly more churners at the cost of more false alarms.

**In practice.**
- Information retrieval and web search are evaluated by precision and recall of the returned documents; the Text REtrieval Conference (TREC, run by the US National Institute of Standards and Technology since 1992) established these measures as a standard.
- Content moderation systems report precision (how many removed posts really violated the rules) and recall (how many violating posts were found) separately, because both errors carry public criticism.

> **Caution.** Precision, recall and F1 depend on which class is called positive. `precision_score` uses `pos_label=1` by default. For string labels such as "Yes"/"No" set `pos_label="Yes"`, or you will get an error or the wrong class.

## Macro-F1 for several classes

**Concept.** With more than two classes, precision, recall and F1 are computed **per class**, treating that class as positive and all others as negative (one-vs-rest). They are then averaged:

- **Macro-F1**: the unweighted mean of the per-class F1 scores. Every class counts equally, however rare.
- **Weighted F1**: weighted by the number of true cases per class; dominated by the large classes.
- **Micro-F1**: pools all decisions; for single-label problems it equals accuracy.

Worked example: a telecom company wants to predict which contract a new customer will choose, to plan its sales staff. Ten customers: six choose month-to-month (M), two a one-year (1Y) and two a two-year contract (2Y). The model gets all six M right, one of the two 1Y and one of the two 2Y; both errors are predicted as M. Per-class F1: M 0.86, 1Y 0.67, 2Y 0.67. Macro-F1 = (0.86 + 0.67 + 0.67)/3 = 0.73; accuracy = 8/10 = 0.80.

Macro-F1 is also useful with two classes: it averages the F1 of "churns" and of "stays", so a model that never flags anybody cannot hide behind the large class.

**Why it matters.** Accuracy is dominated by the frequent classes. Macro-F1 gives a class with 3 cases the same weight as one with 2,000, so a model that ignores the rare classes gets a low macro-F1 even when its accuracy looks fine. Always predicting "stays" reaches 73.5 % accuracy on the Telco customers, but a macro-F1 of only 0.42. The gap between the two numbers grows with the number of rare classes; Session 13 meets a task with more than 1,000 classes, where accuracy and macro-F1 tell very different stories and per-class results and top-k accuracy are needed as well.

**How it works in Python.**

In [8]:
from sklearn.dummy import DummyClassifier

y_true = ["M"] * 6 + ["1Y", "1Y", "2Y", "2Y"]
y_pred = ["M"] * 6 + ["M", "1Y", "M", "2Y"]
print(f1_score(y_true, y_pred, average=None, labels=["M", "1Y", "2Y"]).round(2))   # [0.86 0.67 0.67]
print(round(f1_score(y_true, y_pred, average="macro"), 2))                       # 0.73
print(round(accuracy_score(y_true, y_pred), 2))                                  # 0.8

# two classes: the majority baseline against the logistic regression
always_stays = DummyClassifier(strategy="most_frequent").fit(X_train, y_train).predict(X_test)
for name, p in [("always stays", always_stays), ("logistic regression", pred)]:
    print(name, round(accuracy_score(y_test, p), 3), round(f1_score(y_test, p, average="macro"), 3))
# always stays        0.735 0.424
# logistic regression 0.792 0.718

[0.86 0.67 0.67]
0.73
0.8
always stays 0.735 0.424
logistic regression 0.792 0.718


**In practice.**
- Shared tasks in text classification, such as SemEval-2017 Task 4 (Rosenthal et al., 2017), use macro-averaged measures so that rare classes count.
- Automatic coding of occupations, causes of death or economic activities by statistical offices deals with hundreds of codes and a long tail; evaluations report accuracy together with per-class or macro-averaged results.

> **Warning.** `f1_score` on string labels without `average=` raises an error for more than two classes. Always pass `average="macro"` (or the average you intend) and say which one you report. With very many classes macro-F1 also depends on which classes appear at all in the evaluation set; compare macro-F1 values only on the same set.

> 📖 **Read more:** [extras/05-classification-metrics.ipynb](extras/05-classification-metrics.ipynb) (INRIA scikit-learn MOOC): accuracy, balanced accuracy, the confusion matrix, precision, recall and the ROC and precision–recall curves on the blood transfusion data, with a dummy baseline.

## Check your understanding

1. For each column, choose one-hot or ordinal encoding and justify: `PaymentMethod`, `Contract`, `InternetService`, a satisfaction score from 1 to 5.
2. Explain what happens to the `StandardScaler` inside a pipeline when you call `cross_val_score(pipeline, X, y, cv=5)`.
3. A 1-NN model has a training accuracy of 99.7 %. Why does that not tell you anything about its quality, and how would you choose *k* instead?
4. Compute precision, recall and F1 from TP = 50, FP = 30, FN = 20, TN = 900. What is the accuracy, and why is it misleading here?
5. Why does macro-F1 punish a model that never predicts a rare class, while accuracy hardly notices? Compute both for "always predict stays" on the Telco test customers.

## Further reading

- James, G., Witten, D., Hastie, T., Tibshirani, R. & Taylor, J. (2023). *An Introduction to Statistical Learning with Applications in Python*, Sections 2.2 (k-nearest neighbours) and 4.7 (lab). Springer. https://www.statlearning.com/
- INRIA (2024). *scikit-learn MOOC, Module 1: The predictive modeling pipeline* (categorical encoding, ColumnTransformer). https://inria.github.io/scikit-learn-mooc/
- scikit-learn developers (2025). *Pipelines and composite estimators* and *Metrics and scoring*. https://scikit-learn.org/stable/modules/compose.html · https://scikit-learn.org/stable/modules/model_evaluation.html
- Google for Developers (2025). *Machine Learning Crash Course: Classification* (thresholds, confusion matrix, ROC and AUC). https://developers.google.com/machine-learning/crash-course/classification